# 🧸 ABC Toys — Phân tích & Trực quan hóa Dữ liệu

**Bài thu hoạch cuối khóa — Data Visualization for Data Analysis**

---

## 📋 Context (Bối cảnh giả định)

Nhóm chúng tôi là **Data Analyst Team của ABC Toys** — một chuỗi bán lẻ đồ chơi với 50 cửa hàng tại 29 thành phố. Ban giám đốc đang chuẩn bị họp chiến lược cuối năm và cần một báo cáo phân tích toàn diện về:

- Hiệu quả kinh doanh 21 tháng vừa qua (01/2022 – 09/2023)
- Đâu là điểm sáng, đâu là điểm tối cần can thiệp
- 3 ưu tiên kinh doanh nên đầu tư trong năm tới

**Đối tượng nghe**: Ban giám đốc (CEO, CFO, Head of Retail, Head of Supply Chain)

---

## 🎯 7 Câu hỏi phân tích

1. Doanh thu & lợi nhuận 21 tháng có xu hướng gì? Có tính mùa vụ không?
2. So sánh 9 tháng đầu năm 2023 với 2022 — tăng/giảm bao nhiêu?
3. Category nào sinh lời nhất? (đánh đổi giữa Revenue và Margin)
4. Sản phẩm nào là "ngôi sao", "bò sữa", "ế ẩm"? (BCG-style quadrant)
5. Loại địa điểm (Downtown/Commercial/Residential/Airport) nào hiệu quả nhất?
6. Top thành phố theo doanh thu — có chênh lệch lớn không?
7. Tồn kho có vấn đề gì? Sản phẩm nào hết hàng ở nhiều store?

---

## 1. Setup & Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Cấu hình hiển thị
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print('✓ Libraries loaded')

## 2. Load Data

ABC Toys cung cấp 4 file dữ liệu:
- `sales.csv` — Lịch sử giao dịch
- `stores.csv` — Thông tin 50 cửa hàng
- `products.csv` — Catalog 35 sản phẩm
- `inventory.xlsx` — Tồn kho hiện tại

In [ ]:
# Đọc 4 nguồn dữ liệu
sales = pd.read_csv('sales.csv')
stores = pd.read_csv('stores.csv')
products = pd.read_csv('products.csv')
inventory = pd.read_excel('inventory.xlsx')

print(f'Sales:     {sales.shape[0]:>8,} rows × {sales.shape[1]} cols')
print(f'Stores:    {stores.shape[0]:>8,} rows × {stores.shape[1]} cols')
print(f'Products:  {products.shape[0]:>8,} rows × {products.shape[1]} cols')
print(f'Inventory: {inventory.shape[0]:>8,} rows × {inventory.shape[1]} cols')

In [ ]:
# Xem nhanh từng bảng
print('=== SALES ===')
display(sales.head(3))
print('\n=== STORES ===')
display(stores.head(3))
print('\n=== PRODUCTS ===')
display(products.head(3))
print('\n=== INVENTORY ===')
display(inventory.head(3))

## 3. Data Cleaning & Merging

**Vấn đề cần xử lý:**
- Cột `Date` đang là string → cần convert sang datetime
- Cột `Product_Cost` và `Product_Price` có ký tự `$` và khoảng trắng → cần clean
- Cần merge 3 bảng `sales + products + stores` thành 1 bảng phân tích chính

In [ ]:
# 3.1 Convert Date
sales['Date'] = pd.to_datetime(sales['Date'])

# 3.2 Clean price columns
products['Product_Cost']  = products['Product_Cost'].str.replace('$','', regex=False).str.strip().astype(float)
products['Product_Price'] = products['Product_Price'].str.replace('$','', regex=False).str.strip().astype(float)

# Tính sẵn margin
products['Profit_Per_Unit'] = products['Product_Price'] - products['Product_Cost']
products['Margin_Pct']      = products['Profit_Per_Unit'] / products['Product_Price'] * 100

# 3.3 Convert Store_Open_Date
stores['Store_Open_Date'] = pd.to_datetime(stores['Store_Open_Date'])

# 3.4 Kiểm tra missing values
print('Missing values:')
for name, df in [('sales',sales),('stores',stores),('products',products),('inventory',inventory)]:
    print(f'  {name}: {df.isnull().sum().sum()} missing values')

In [ ]:
# 3.5 Merge thành bảng phân tích chính
df = (sales
      .merge(products, on='Product_ID', how='left')
      .merge(stores,   on='Store_ID',   how='left'))

# Tạo các trường tính toán
df['Revenue'] = df['Units'] * df['Product_Price']
df['Profit']  = df['Units'] * df['Profit_Per_Unit']
df['Year']    = df['Date'].dt.year
df['Month']   = df['Date'].dt.month
df['YearMonth'] = df['Date'].dt.to_period('M').astype(str)
df['Quarter']   = df['Date'].dt.quarter
df['Weekday']   = df['Date'].dt.day_name()

print(f'Bảng merged: {df.shape[0]:,} rows × {df.shape[1]} cols')
print(f'Date range: {df.Date.min().date()} → {df.Date.max().date()}')
df.head()

## 4. EDA — Headline Numbers

Trước khi đi sâu, ta xem các con số tổng quan để có "bức tranh lớn".

In [ ]:
# Headline KPIs
total_rev   = df['Revenue'].sum()
total_prof  = df['Profit'].sum()
total_units = df['Units'].sum()
margin_pct  = total_prof / total_rev * 100
n_orders    = len(df)

print('=' * 50)
print('  ABC TOYS — HEADLINE METRICS (21 tháng)')
print('=' * 50)
print(f'  Total Revenue : ${total_rev:>15,.0f}')
print(f'  Total Profit  : ${total_prof:>15,.0f}')
print(f'  Overall Margin: {margin_pct:>15.1f}%')
print(f'  Units Sold    : {total_units:>15,}')
print(f'  Transactions  : {n_orders:>15,}')
print(f'  Avg Units/Tx  : {total_units/n_orders:>15.2f}')
print('=' * 50)

## 5. Q1 — Doanh thu & Lợi nhuận có xu hướng gì? Có mùa vụ không?

📊 **Biểu đồ**: Line chart kép — Revenue và Profit theo tháng

In [ ]:
# Aggregate theo tháng
monthly = df.groupby('YearMonth').agg(
    Revenue=('Revenue','sum'),
    Profit =('Profit','sum'),
    Units  =('Units','sum')
).reset_index()
monthly['Margin_Pct'] = monthly['Profit']/monthly['Revenue']*100

fig, ax1 = plt.subplots(figsize=(14, 6))

# Revenue & Profit lines
ax1.plot(monthly['YearMonth'], monthly['Revenue']/1000, marker='o', linewidth=2.5,
         color='#2E86AB', label='Revenue ($K)')
ax1.plot(monthly['YearMonth'], monthly['Profit']/1000,  marker='s', linewidth=2.5,
         color='#A23B72', label='Profit ($K)')
ax1.fill_between(monthly['YearMonth'], 0, monthly['Revenue']/1000, alpha=0.1, color='#2E86AB')

ax1.set_xlabel('Tháng', fontsize=12)
ax1.set_ylabel('USD (nghìn)', fontsize=12)
ax1.set_title('Q1 — Doanh thu & Lợi nhuận theo tháng (01/2022 – 09/2023)', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left', fontsize=11)
ax1.tick_params(axis='x', rotation=45)
ax1.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\n📌 Insights:')
print(f'  • Tháng cao nhất: {monthly.loc[monthly.Revenue.idxmax(),"YearMonth"]} (${monthly.Revenue.max():,.0f})')
print(f'  • Tháng thấp nhất: {monthly.loc[monthly.Revenue.idxmin(),"YearMonth"]} (${monthly.Revenue.min():,.0f})')
print(f'  • Chênh lệch peak/low: {monthly.Revenue.max()/monthly.Revenue.min():.1f}x')

**📊 Bổ sung**: Heatmap mùa vụ — Tháng × Năm

In [ ]:
# Heatmap để nhìn pattern theo mùa
season = df.groupby(['Year','Month'])['Revenue'].sum().unstack()

plt.figure(figsize=(13, 4))
sns.heatmap(season/1000, annot=True, fmt=',.0f', cmap='YlOrRd',
            cbar_kws={'label':'Revenue ($K)'}, linewidths=0.5)
plt.title('Q1 — Mùa vụ: Doanh thu theo Tháng × Năm ($K)', fontweight='bold')
plt.xlabel('Tháng')
plt.ylabel('Năm')
plt.tight_layout()
plt.show()

print('\n📌 Insights mùa vụ:')
print('  • Tháng 5 và tháng 11–12 thường là cao điểm (lễ + cuối năm)')
print('  • Tháng 9 thường là tháng thấp điểm sau back-to-school')

## 6. Q2 — So sánh 2023 vs 2022 (9 tháng đầu năm)

📊 **Biểu đồ**: Bar chart cạnh nhau

In [ ]:
# Lọc cùng 9 tháng (Jan-Sep) cho fair comparison
df_9m = df[df['Month'] <= 9]

yoy = df_9m.groupby('Year').agg(
    Revenue=('Revenue','sum'),
    Profit =('Profit','sum'),
    Units  =('Units','sum')
).reset_index()

# Tính % thay đổi
rev_change  = (yoy.Revenue.iloc[1] - yoy.Revenue.iloc[0]) / yoy.Revenue.iloc[0] * 100
prof_change = (yoy.Profit.iloc[1] - yoy.Profit.iloc[0]) / yoy.Profit.iloc[0] * 100
unit_change = (yoy.Units.iloc[1] - yoy.Units.iloc[0]) / yoy.Units.iloc[0] * 100

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics = [('Revenue', 1000, '$K', rev_change),
           ('Profit',  1000, '$K', prof_change),
           ('Units',   1,    '',   unit_change)]
colors = ['#2E86AB', '#A23B72', '#F18F01']

for ax, (col, div, unit, chg), color in zip(axes, metrics, colors):
    bars = ax.bar(yoy['Year'].astype(str), yoy[col]/div, color=color, edgecolor='black', linewidth=1.2)
    for bar, val in zip(bars, yoy[col]/div):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
                f'{val:,.0f}{unit}', ha='center', fontweight='bold', fontsize=11)
    arrow = '▲' if chg >= 0 else '▼'
    color_chg = 'green' if chg >= 0 else 'red'
    ax.set_title(f'{col} (9 tháng đầu năm)\n{arrow} {abs(chg):.1f}% YoY',
                 fontweight='bold', color=color_chg)
    ax.set_ylabel(f'{col} ({unit})' if unit else col)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Q2 — So sánh 2023 vs 2022 (Jan–Sep)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'\n📌 Insights YoY:')
print(f'  • Revenue: {rev_change:+.1f}%')
print(f'  • Profit:  {prof_change:+.1f}%')
print(f'  • Units:   {unit_change:+.1f}%')

## 7. Q3 — Category nào sinh lời nhất?

📊 **Biểu đồ**: Scatter plot — Revenue (x) × Margin% (y), size = Units

In [ ]:
# Aggregate theo category
cat = df.groupby('Product_Category').agg(
    Revenue=('Revenue','sum'),
    Profit =('Profit','sum'),
    Units  =('Units','sum')
).reset_index()
cat['Margin_Pct'] = cat['Profit']/cat['Revenue']*100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Stacked bar Revenue vs Profit
cat_sorted = cat.sort_values('Revenue', ascending=True)
y_pos = np.arange(len(cat_sorted))
axes[0].barh(y_pos, cat_sorted['Revenue']/1000, color='#2E86AB', label='Revenue', alpha=0.85)
axes[0].barh(y_pos, cat_sorted['Profit']/1000,  color='#A23B72', label='Profit',  alpha=0.95)
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(cat_sorted['Product_Category'])
axes[0].set_xlabel('USD (nghìn)')
axes[0].set_title('Revenue vs Profit theo Category', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='x')
for i, (rev, prof) in enumerate(zip(cat_sorted['Revenue']/1000, cat_sorted['Profit']/1000)):
    axes[0].text(rev*1.01, i, f'${rev:,.0f}K', va='center', fontsize=10)

# Right: Scatter Revenue vs Margin (bubble = Units)
scatter = axes[1].scatter(cat['Revenue']/1000, cat['Margin_Pct'],
                          s=cat['Units']/300, alpha=0.65,
                          c=range(len(cat)), cmap='Set2', edgecolors='black', linewidth=1.5)
for _, row in cat.iterrows():
    axes[1].annotate(row['Product_Category'],
                     (row['Revenue']/1000, row['Margin_Pct']),
                     xytext=(8, 8), textcoords='offset points', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Revenue ($K)')
axes[1].set_ylabel('Margin (%)')
axes[1].set_title('Bubble: Revenue × Margin × Units sold', fontweight='bold')
axes[1].axhline(cat['Margin_Pct'].mean(), color='red', linestyle='--', alpha=0.5, label=f'Avg margin: {cat.Margin_Pct.mean():.1f}%')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.suptitle('Q3 — Phân tích Category Performance', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

display(cat.sort_values('Revenue', ascending=False).round(2))

print('\n📌 Insights:')
print('  • Toys: doanh thu lớn nhất nhưng margin trung bình')
print('  • Electronics: margin cao nhất (~45%) — đáng đầu tư!')
print('  • Art & Crafts: volume cao nhất (units) nhưng giá thấp')

## 8. Q4 — Phân loại sản phẩm theo ma trận Revenue × Margin

📊 **Biểu đồ**: BCG-style quadrant — chia 4 góc phần tư

- **⭐ Stars** (Rev cao, Margin cao): đầu tư mạnh
- **🐄 Cash Cows** (Rev cao, Margin thấp): khai thác ổn định
- **❓ Question Marks** (Rev thấp, Margin cao): có thể đẩy mạnh
- **🐕 Dogs** (Rev thấp, Margin thấp): cân nhắc loại bỏ

In [ ]:
# Aggregate theo product
prod = df.groupby(['Product_Name','Product_Category']).agg(
    Revenue=('Revenue','sum'),
    Profit =('Profit','sum'),
    Units  =('Units','sum')
).reset_index()
prod['Margin_Pct'] = prod['Profit']/prod['Revenue']*100

# Median lines để chia quadrant
rev_med = prod['Revenue'].median()
mar_med = prod['Margin_Pct'].median()

# Gán nhãn quadrant
def quadrant(r):
    if r['Revenue'] >= rev_med and r['Margin_Pct'] >= mar_med: return '⭐ Star'
    if r['Revenue'] >= rev_med and r['Margin_Pct'] <  mar_med: return '🐄 Cash Cow'
    if r['Revenue'] <  rev_med and r['Margin_Pct'] >= mar_med: return '❓ Question'
    return '🐕 Dog'
prod['Quadrant'] = prod.apply(quadrant, axis=1)

# Plot
fig, ax = plt.subplots(figsize=(14, 9))
colors_map = {'⭐ Star':'#2ECC71', '🐄 Cash Cow':'#3498DB',
              '❓ Question':'#F39C12', '🐕 Dog':'#E74C3C'}
for q, color in colors_map.items():
    sub = prod[prod['Quadrant']==q]
    ax.scatter(sub['Revenue']/1000, sub['Margin_Pct'], s=sub['Units']/50,
               alpha=0.7, c=color, label=q, edgecolors='black', linewidth=1)

# Median lines
ax.axhline(mar_med, color='gray', linestyle='--', alpha=0.6)
ax.axvline(rev_med/1000, color='gray', linestyle='--', alpha=0.6)

# Label products
for _, row in prod.iterrows():
    ax.annotate(row['Product_Name'], (row['Revenue']/1000, row['Margin_Pct']),
                xytext=(4, 4), textcoords='offset points', fontsize=8, alpha=0.85)

ax.set_xlabel('Revenue ($K)', fontsize=12)
ax.set_ylabel('Margin (%)', fontsize=12)
ax.set_title('Q4 — Product Quadrant Matrix (Revenue × Margin)\nSize = Units Sold',
             fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=11, framealpha=0.95)
ax.grid(True, alpha=0.3)
ax.set_xscale('log')
plt.tight_layout()
plt.show()

# In ra danh sách từng quadrant
print('\n📌 Phân loại sản phẩm:')
for q in ['⭐ Star','🐄 Cash Cow','❓ Question','🐕 Dog']:
    items = prod[prod['Quadrant']==q]['Product_Name'].tolist()
    print(f'\n  {q} ({len(items)} sản phẩm):')
    for it in items: print(f'      • {it}')

## 9. Q5 — Loại địa điểm nào hiệu quả nhất per store?

📊 **Biểu đồ**: Bar chart so sánh Revenue **tuyệt đối** vs Revenue **per store**

In [ ]:
loc = df.groupby('Store_Location').agg(
    Revenue =('Revenue','sum'),
    Profit  =('Profit','sum')
).reset_index()
# Đếm số store mỗi loại
loc['Num_Stores'] = loc['Store_Location'].map(stores.groupby('Store_Location').size())
loc['Revenue_per_Store'] = loc['Revenue'] / loc['Num_Stores']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: Total Revenue
loc_s = loc.sort_values('Revenue', ascending=True)
bars1 = axes[0].barh(loc_s['Store_Location'], loc_s['Revenue']/1000, color='#2E86AB', edgecolor='black')
for b, n in zip(bars1, loc_s['Num_Stores']):
    axes[0].text(b.get_width()*1.01, b.get_y()+b.get_height()/2,
                 f'  {b.get_width():,.0f}K  ({n} stores)', va='center', fontsize=10)
axes[0].set_xlabel('Total Revenue ($K)')
axes[0].set_title('Tổng Revenue theo Loại địa điểm', fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')

# Right: Revenue per Store (chuẩn hóa)
loc_s2 = loc.sort_values('Revenue_per_Store', ascending=True)
bars2 = axes[1].barh(loc_s2['Store_Location'], loc_s2['Revenue_per_Store']/1000,
                     color='#A23B72', edgecolor='black')
for b in bars2:
    axes[1].text(b.get_width()*1.01, b.get_y()+b.get_height()/2,
                 f'  ${b.get_width():,.0f}K/store', va='center', fontsize=10)
axes[1].set_xlabel('Revenue per Store ($K)')
axes[1].set_title('Revenue chuẩn hóa per Store ⭐', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')

plt.suptitle('Q5 — Hiệu quả theo Loại địa điểm', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

display(loc.round(0))

print('\n📌 Insight quan trọng:')
print('  Khi nhìn TOTAL → Downtown vượt trội (vì có 29 stores)')
print('  Khi nhìn PER STORE → có thể Airport hiệu quả nhất (ít store nhưng mỗi store khỏe)')
print('  → Bài học: phải normalize trước khi so sánh!')

## 10. Q6 — Top thành phố theo doanh thu

📊 **Biểu đồ**: Horizontal bar chart — Top 15 cities

In [ ]:
city = df.groupby('Store_City').agg(
    Revenue   =('Revenue','sum'),
    Profit    =('Profit','sum'),
    Num_Stores=('Store_ID','nunique')
).reset_index().sort_values('Revenue', ascending=False)
city['Revenue_per_Store'] = city['Revenue']/city['Num_Stores']

top15 = city.head(15).sort_values('Revenue', ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(top15['Store_City'], top15['Revenue']/1000,
               color=plt.cm.viridis(np.linspace(0.3, 0.9, len(top15))),
               edgecolor='black')
for b, ns in zip(bars, top15['Num_Stores']):
    ax.text(b.get_width()*1.01, b.get_y()+b.get_height()/2,
            f'  ${b.get_width():,.0f}K ({ns} store{"s" if ns>1 else ""})',
            va='center', fontsize=10)
ax.set_xlabel('Revenue ($K)', fontsize=12)
ax.set_title('Q6 — Top 15 thành phố theo Doanh thu', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print('\n📌 Insights:')
print(f'  • Top 1: {city.iloc[0].Store_City} (${city.iloc[0].Revenue:,.0f})')
print(f'  • Top 5 cities chiếm {city.head(5).Revenue.sum()/city.Revenue.sum()*100:.1f}% tổng revenue')
print(f'  • Số thành phố: {len(city)}')
print('  → Tập trung 80/20: ít thành phố tạo phần lớn doanh thu')

## 11. Q7 — Tình hình tồn kho

📊 **Biểu đồ**: Heatmap Store × Product + Bar chart sản phẩm hết hàng

In [ ]:
# Số store đang out-of-stock cho mỗi product
oos = inventory[inventory['Stock_On_Hand']==0]
oos_by_product = oos.groupby('Product_ID').size().reset_index(name='Stores_OOS')
oos_by_product = oos_by_product.merge(products[['Product_ID','Product_Name','Product_Category']], on='Product_ID')
oos_by_product = oos_by_product.sort_values('Stores_OOS', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Left: Heatmap tồn kho (top 20 products * all stores)
top_products = df.groupby('Product_ID')['Units'].sum().nlargest(20).index
inv_pivot = inventory[inventory['Product_ID'].isin(top_products)].pivot(
    index='Product_ID', columns='Store_ID', values='Stock_On_Hand')
inv_pivot.index = [products.set_index('Product_ID').loc[i,'Product_Name'] for i in inv_pivot.index]

sns.heatmap(inv_pivot, cmap='RdYlGn', center=20, ax=axes[0],
            cbar_kws={'label':'Stock On Hand'}, linewidths=0.3)
axes[0].set_title('Tồn kho: Top 20 Products × 50 Stores', fontweight='bold')
axes[0].set_xlabel('Store ID')
axes[0].set_ylabel('Product')

# Right: Top products đang OOS ở nhiều store
top_oos = oos_by_product.head(15)
colors_cat = {'Toys':'#FF6B6B', 'Art & Crafts':'#4ECDC4', 'Games':'#FFE66D',
              'Electronics':'#A8E6CF', 'Sports & Outdoors':'#C7B8EA'}
bar_colors = [colors_cat[c] for c in top_oos['Product_Category']]

axes[1].barh(top_oos['Product_Name'], top_oos['Stores_OOS'],
             color=bar_colors, edgecolor='black')
axes[1].set_xlabel('Số store đang HẾT HÀNG')
axes[1].set_title('Top 15 sản phẩm hết hàng ở nhiều store', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')
axes[1].invert_yaxis()

plt.suptitle('Q7 — Phân tích Tồn kho', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\n📌 Insights tồn kho:')
print(f'  • Tổng cặp Store-Product đang OOS: {len(oos)} (= {len(oos)/len(inventory)*100:.1f}%)')
print(f'  • Sản phẩm OOS ở nhiều store nhất: {oos_by_product.iloc[0].Product_Name} ({oos_by_product.iloc[0].Stores_OOS} stores)')
print(f'  • → Đây có thể là supply chain issue, ảnh hưởng trực tiếp đến doanh thu')

## 12. 📋 Executive Summary & Recommendations

### 🎯 Key Findings (Tổng hợp 7 phân tích)

| # | Phát hiện chính | Mức độ |
|---|---|---|
| 1 | Doanh thu tăng trưởng 21 tháng, có **2 đỉnh mùa vụ** (tháng 5 và tháng 11–12) | 🟢 Tích cực |
| 2 | 2023 vs 2022 tăng trưởng dương ở cả Revenue/Profit/Units | 🟢 Tích cực |
| 3 | **Electronics** có margin cao nhất nhưng revenue chưa tương xứng | 🟡 Cơ hội |
| 4 | **Lego Bricks** là Cash Cow (revenue lớn, margin thấp 12.5%) — cần xem lại pricing | 🟡 Cảnh báo |
| 5 | **Airport stores** ít nhưng efficiency per store có thể cao nhất | 🟢 Tiềm năng |
| 6 | Top 5 thành phố chiếm ~50% doanh thu — phụ thuộc cao | 🟡 Rủi ro |
| 7 | **77 cặp Store-Product hết hàng** (4.8% inventory) | 🔴 Cần xử lý ngay |

### 💡 3 Recommendations cho 2024

**1️⃣ Đầu tư mạnh vào danh mục Electronics** (Star quadrant)
- Margin trung bình ~45% — gấp đôi Toys
- Tăng SKU, marketing, training nhân viên về sản phẩm Electronics
- Mục tiêu: đẩy revenue Electronics lên ngang Toys trong 12 tháng

**2️⃣ Xem lại pricing/sourcing cho các "Cash Cow"**
- Lego Bricks margin chỉ 12.5% — đàm phán lại với nhà cung cấp
- Hoặc nâng giá nhẹ 5–10% (test A/B vài store)
- Tiềm năng: tăng $200K+ profit/năm chỉ từ Lego

**3️⃣ Giải quyết khẩn cấp vấn đề tồn kho**
- Thiết lập alert tự động khi stock < safety level
- Re-allocate hàng giữa các store (heatmap cho thấy chỗ thừa, chỗ thiếu)
- Phối hợp với Supply Chain để dự báo demand chính xác hơn

---

### 📊 Next Steps cho nhóm

- ✅ Notebook EDA này (đã hoàn thành)
- ➡️ Tableau Dashboard interactive (cho phần demo trực tiếp)
- ➡️ Slides PPT thuyết trình (15 phút)

---

*Báo cáo phân tích bởi Data Analyst Team — ABC Toys, 2024*